<!-- #Updateing 2.0 -->

In [ ]:
import os
import re
import math
import time
import random
import warnings
from collections import Counter

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

from sklearn.pipeline import Pipeline
from sklearn.decomposition import TruncatedSVD

from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    HistGradientBoostingClassifier
)
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.neural_network import MLPClassifier



# SETTINGS

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

FAST_MODE = False

if FAST_MODE:
    MAX_ML_SAMPLES = 30000
    SAMPLES_PER_SEQUENCE = 2
    RF_TREES = 100
    ET_TREES = 100
    MLP_MAX_ITER = 150
    GAP_EVAL_ROWS = 200
else:
    MAX_ML_SAMPLES = 150000
    SAMPLES_PER_SEQUENCE = 5
    RF_TREES = 300
    ET_TREES = 300
    MLP_MAX_ITER = 300
    GAP_EVAL_ROWS = 1000

TOP_N_MODELS = 3
KMER_SIZE = 11
BEAM_WIDTH = 5
RESIDUE_TOP_K = 5

OUTPUT_DIR = "/kaggle/working"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("==== Settings ====")
print("FAST_MODE:", FAST_MODE)
print("MAX_ML_SAMPLES:", MAX_ML_SAMPLES)
print("KMER_SIZE:", KMER_SIZE)
print("TOP_N_MODELS:", TOP_N_MODELS)
print("BEAM_WIDTH:", BEAM_WIDTH)
print("RESIDUE_TOP_K:", RESIDUE_TOP_K)



# DATASET PATHS

DATA_DIR = "/kaggle/input/datasets/tahmidenamshrestha/protei-scaffold-datasets"

DE_NOVO_CAH2 = f"{DATA_DIR}/de_novo_sequence_CAH2.txt"
MAB_TARGET = f"{DATA_DIR}/mab_target_sequence.txt"
MAB_TRAINING = f"{DATA_DIR}/mab_training_sequence.txt"
P5A_TRAINING = f"{DATA_DIR}/p5A_training_sequence.txt"
CAH2_TARGET = f"{DATA_DIR}/target_sequence_CAH2.txt"
CAH2_TRAINING = f"{DATA_DIR}/training_sequences_CAH2.txt"


def find_file(filename, root="/kaggle/input"):
    for dirpath, _, files in os.walk(root):
        if filename in files:
            return os.path.join(dirpath, filename)
    return None


paths = {
    "DE_NOVO_CAH2": ("de_novo_sequence_CAH2.txt", DE_NOVO_CAH2),
    "MAB_TARGET": ("mab_target_sequence.txt", MAB_TARGET),
    "MAB_TRAINING": ("mab_training_sequence.txt", MAB_TRAINING),
    "P5A_TRAINING": ("p5A_training_sequence.txt", P5A_TRAINING),
    "CAH2_TARGET": ("target_sequence_CAH2.txt", CAH2_TARGET),
    "CAH2_TRAINING": ("training_sequences_CAH2.txt", CAH2_TRAINING),
}

fixed_paths = {}

for key, (fname, path) in paths.items():
    if os.path.exists(path):
        fixed_paths[key] = path
    else:
        fixed_paths[key] = find_file(fname)

DE_NOVO_CAH2 = fixed_paths["DE_NOVO_CAH2"]
MAB_TARGET = fixed_paths["MAB_TARGET"]
MAB_TRAINING = fixed_paths["MAB_TRAINING"]
P5A_TRAINING = fixed_paths["P5A_TRAINING"]
CAH2_TARGET = fixed_paths["CAH2_TARGET"]
CAH2_TRAINING = fixed_paths["CAH2_TRAINING"]

print("\n==== Dataset Paths ====")
for name, path in fixed_paths.items():
    print(name, "=>", path, "| exists:", path is not None and os.path.exists(path))


# AMINO ACID VOCABULARY, MASS TABLE, AND BIOLOGICAL METRICS

AMINO_ACIDS = list("ACDEFGHIKLMNPQRSTVWY")
GAP_TOKEN = "-"

AA_MASS = {
    "G": 57, "A": 71, "S": 87, "P": 97, "V": 99,
    "T": 101, "C": 103, "L": 113, "I": 113, "N": 114,
    "D": 115, "Q": 128, "K": 128, "E": 129, "M": 131,
    "H": 137, "F": 147, "R": 156, "Y": 163, "W": 186
}

aa_to_int = {aa: i + 1 for i, aa in enumerate(AMINO_ACIDS)}
aa_to_int[GAP_TOKEN] = 0
int_to_aa = {v: k for k, v in aa_to_int.items()}


def peptide_mass(seq):
    return int(sum(AA_MASS.get(aa, 0) for aa in seq))


def is_valid_protein_sequence(seq):
    return all(ch in AMINO_ACIDS for ch in seq)


def encode_sequence(seq):
    return [aa_to_int.get(ch, 0) for ch in seq]


AA_HYDROPHOBICITY = {
    "I": 4.5, "V": 4.2, "L": 3.8, "F": 2.8, "C": 2.5,
    "M": 1.9, "A": 1.8, "G": -0.4, "T": -0.7, "S": -0.8,
    "W": -0.9, "Y": -1.3, "P": -1.6, "H": -3.2,
    "E": -3.5, "Q": -3.5, "D": -3.5, "N": -3.5,
    "K": -3.9, "R": -4.5
}

AA_CHARGE = {
    "K": 1.0,
    "R": 1.0,
    "H": 0.1,
    "D": -1.0,
    "E": -1.0
}


def mean_hydrophobicity(seq):
    if len(seq) == 0:
        return 0.0
    return float(np.mean([AA_HYDROPHOBICITY.get(aa, 0.0) for aa in seq]))


def net_charge(seq):
    return float(sum(AA_CHARGE.get(aa, 0.0) for aa in seq))


def aa_composition_vector(seq):
    counts = np.array([seq.count(aa) for aa in AMINO_ACIDS], dtype=float)
    total = counts.sum()
    if total == 0:
        return counts
    return counts / total


def composition_cosine_similarity(seq1, seq2):
    v1 = aa_composition_vector(seq1)
    v2 = aa_composition_vector(seq2)
    denom = np.linalg.norm(v1) * np.linalg.norm(v2)

    if denom == 0:
        return 0.0

    return float(np.dot(v1, v2) / denom)


try:
    from Bio.Align import substitution_matrices
    BLOSUM62 = substitution_matrices.load("BLOSUM62")
    BIOPYTHON_AVAILABLE = True
except Exception:
    BLOSUM62 = None
    BIOPYTHON_AVAILABLE = False


def blosum62_similarity(seq1, seq2):
    if len(seq1) == 0 or len(seq2) == 0:
        return 0.0

    n = min(len(seq1), len(seq2))
    scores = []

    for a, b in zip(seq1[:n], seq2[:n]):
        if BIOPYTHON_AVAILABLE:
            try:
                scores.append(float(BLOSUM62[a, b]))
            except Exception:
                scores.append(0.0)
        else:
            scores.append(1.0 if a == b else -1.0)

    return float(np.mean(scores)) if scores else 0.0


def biochemical_validation_metrics(pred, truth):
    return {
        "mass_error": abs(peptide_mass(pred) - peptide_mass(truth)),
        "hydrophobicity_truth": mean_hydrophobicity(truth),
        "hydrophobicity_pred": mean_hydrophobicity(pred),
        "hydrophobicity_diff": abs(mean_hydrophobicity(pred) - mean_hydrophobicity(truth)),
        "charge_truth": net_charge(truth),
        "charge_pred": net_charge(pred),
        "charge_diff": abs(net_charge(pred) - net_charge(truth)),
        "composition_similarity": composition_cosine_similarity(pred, truth),
        "blosum62_similarity": blosum62_similarity(pred, truth)
    }


print("\n==== Amino Acid Checks ====")
print("Amino acids:", AMINO_ACIDS)
print("Mass GPEH:", peptide_mass("GPEH"))
print("Mass SVSYDQA:", peptide_mass("SVSYDQA"))
print("Biopython BLOSUM62 available:", BIOPYTHON_AVAILABLE)


# FASTA / TEXT PARSER AND SEQUENCE LOADING

def read_text(path):
    if path is None or not os.path.exists(path):
        raise FileNotFoundError(f"File not found: {path}")
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        return f.read()


def clean_sequence(seq):
    seq = seq.upper()
    seq = re.sub(r"[^ACDEFGHIKLMNPQRSTVWY]", "", seq)
    return seq


def parse_fasta_or_sequences(path):
    text = read_text(path)
    sequences = []
    current = []

    for line in text.splitlines():
        line = line.strip()

        if not line:
            continue

        if line.startswith(">"):
            if current:
                seq = clean_sequence("".join(current))
                if seq:
                    sequences.append(seq)
                current = []
        else:
            current.append(line)

    if current:
        seq = clean_sequence("".join(current))
        if seq:
            sequences.append(seq)

    return sequences


mab_target_seq = parse_fasta_or_sequences(MAB_TARGET)[0]
mab_train_seqs = parse_fasta_or_sequences(MAB_TRAINING)

p5a_train_seqs = parse_fasta_or_sequences(P5A_TRAINING)

cah2_target_seq = parse_fasta_or_sequences(CAH2_TARGET)[0]
cah2_train_seqs = parse_fasta_or_sequences(CAH2_TRAINING)

all_homologs = mab_train_seqs + p5a_train_seqs + cah2_train_seqs

print("\n==== Loaded Sequences ====")
print("Mab target length:", len(mab_target_seq))
print("Mab training sequences:", len(mab_train_seqs))
print("P5A training sequences:", len(p5a_train_seqs))
print("CAH2 target length:", len(cah2_target_seq))
print("CAH2 training sequences:", len(cah2_train_seqs))
print("All homolog sequences:", len(all_homologs))


# MASKED K-MER DATASET GENERATION

def generate_masked_kmer_dataset(
    sequences,
    k=11,
    max_samples=150000,
    random_internal_masks=0
):
    if k % 2 == 0:
        raise ValueError("k must be odd, for example 11, 15, or 21.")

    X = []
    y_letters = []
    middle_pos = k // 2

    for seq in sequences:
        if len(seq) < k:
            continue

        for i in range(len(seq) - k + 1):
            kmer = seq[i:i + k]

            if not is_valid_protein_sequence(kmer):
                continue

            mask_positions = {0, middle_pos, k - 1}

            if random_internal_masks > 0:
                possible_positions = list(range(1, k - 1))
                sampled_positions = random.sample(
                    possible_positions,
                    k=min(random_internal_masks, len(possible_positions))
                )
                mask_positions.update(sampled_positions)

            for pos in mask_positions:
                masked = list(kmer)
                true_aa = masked[pos]
                masked[pos] = GAP_TOKEN

                X.append(encode_sequence("".join(masked)))
                y_letters.append(true_aa)

                if max_samples is not None and len(X) >= max_samples:
                    return np.array(X), np.array(y_letters)

    return np.array(X), np.array(y_letters)


X_ml, y_ml_letters = generate_masked_kmer_dataset(
    all_homologs,
    k=KMER_SIZE,
    max_samples=MAX_ML_SAMPLES,
    random_internal_masks=0
)

label_encoder = LabelEncoder()
y_ml = label_encoder.fit_transform(y_ml_letters)

X_train_ml, X_val_ml, y_train_ml, y_val_ml = train_test_split(
    X_ml,
    y_ml,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y_ml
)

print("\n==== ML Dataset ====")
print("X_ml:", X_ml.shape)
print("y_ml:", y_ml.shape)
print("Classes:", list(label_encoder.classes_))
print("Train:", X_train_ml.shape)
print("Validation:", X_val_ml.shape)


def row_average_features(X):
    return X.mean(axis=1).reshape(-1, 1)


X_train_row = row_average_features(X_train_ml)
X_val_row = row_average_features(X_val_ml)

print("\n==== Feature Shapes ====")
print("Raw feature shape:", X_train_ml.shape)
print("Row-average shape:", X_train_row.shape)


# ML MODELS AND TRAINING HELPERS

def get_base_8_models():
    models = {
        "kNN": KNeighborsClassifier(
            n_neighbors=5,
            weights="distance"
        ),

        "DecisionTree": DecisionTreeClassifier(
            max_depth=None,
            min_samples_split=2,
            min_samples_leaf=1,
            class_weight="balanced",
            random_state=RANDOM_STATE
        ),

        "RandomForest": RandomForestClassifier(
            n_estimators=RF_TREES,
            max_depth=None,
            min_samples_split=2,
            min_samples_leaf=1,
            max_features="sqrt",
            class_weight="balanced",
            random_state=RANDOM_STATE,
            n_jobs=-1
        ),

        "ExtraTrees": ExtraTreesClassifier(
            n_estimators=ET_TREES,
            max_depth=None,
            min_samples_split=2,
            min_samples_leaf=1,
            max_features="sqrt",
            class_weight="balanced",
            random_state=RANDOM_STATE,
            n_jobs=-1
        ),

        "HistGradientBoosting": HistGradientBoostingClassifier(
            max_iter=300,
            learning_rate=0.04,
            l2_regularization=0.01,
            random_state=RANDOM_STATE
        ),

        "LogisticRegression": Pipeline([
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(
                max_iter=2000,
                class_weight="balanced",
                n_jobs=-1,
                random_state=RANDOM_STATE
            ))
        ]),

        "LinearSVC": Pipeline([
            ("scaler", StandardScaler()),
            ("clf", LinearSVC(
                class_weight="balanced",
                random_state=RANDOM_STATE
            ))
        ]),

        "MLP": Pipeline([
            ("scaler", StandardScaler()),
            ("clf", MLPClassifier(
                hidden_layer_sizes=(200, 100),
                activation="tanh",
                alpha=0.0005,
                max_iter=MLP_MAX_ITER,
                random_state=RANDOM_STATE
            ))
        ])
    }

    return models


def safe_predict_proba(model, X, num_classes):
    if hasattr(model, "predict_proba"):
        proba = model.predict_proba(X)

        if proba.shape[1] != num_classes:
            fixed = np.zeros((proba.shape[0], num_classes))
            fixed[:, :proba.shape[1]] = proba
            return fixed

        return proba

    if hasattr(model, "decision_function"):
        scores = model.decision_function(X)

        if scores.ndim == 1:
            scores = np.vstack([-scores, scores]).T

        scores = scores - np.max(scores, axis=1, keepdims=True)
        exp_scores = np.exp(scores)
        return exp_scores / exp_scores.sum(axis=1, keepdims=True)

    preds = model.predict(X)
    proba = np.zeros((len(preds), num_classes))

    for i, p in enumerate(preds):
        proba[i, int(p)] = 1.0

    return proba


def calculate_metrics(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)

    return {
        "accuracy": acc,
        "error_rate": 1.0 - acc,
        "precision_macro": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "recall_macro": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "precision_weighted": precision_score(y_true, y_pred, average="weighted", zero_division=0),
        "recall_weighted": recall_score(y_true, y_pred, average="weighted", zero_division=0),
        "f1_weighted": f1_score(y_true, y_pred, average="weighted", zero_division=0)
    }


def train_and_evaluate_models(models, X_train, X_val, y_train, y_val, feature_type="raw"):
    results = []
    fitted_models = {}
    predictions = {}

    for name, model in models.items():
        print(f"\nTraining {feature_type}__{name}...")

        start_time = time.time()

        try:
            model.fit(X_train, y_train)

            train_pred = model.predict(X_train)
            val_pred = model.predict(X_val)

            train_metrics = calculate_metrics(y_train, train_pred)
            val_metrics = calculate_metrics(y_val, val_pred)

            elapsed = time.time() - start_time
            model_key = f"{feature_type}__{name}"

            results.append({
                "model_name": model_key,
                "base_model": name,
                "feature_type": feature_type,
                "train_accuracy": train_metrics["accuracy"],
                "train_error_rate": train_metrics["error_rate"],
                "validation_accuracy": val_metrics["accuracy"],
                "validation_error_rate": val_metrics["error_rate"],
                "precision_macro": val_metrics["precision_macro"],
                "recall_macro": val_metrics["recall_macro"],
                "f1_macro": val_metrics["f1_macro"],
                "precision_weighted": val_metrics["precision_weighted"],
                "recall_weighted": val_metrics["recall_weighted"],
                "f1_weighted": val_metrics["f1_weighted"],
                "train_time_sec": elapsed
            })

            fitted_models[model_key] = model
            predictions[model_key] = val_pred

            print(f"{model_key}")
            print(f"Train accuracy       : {train_metrics['accuracy']:.4f}")
            print(f"Validation accuracy  : {val_metrics['accuracy']:.4f}")
            print(f"Validation error     : {val_metrics['error_rate']:.4f}")
            print(f"Precision macro      : {val_metrics['precision_macro']:.4f}")
            print(f"Recall macro         : {val_metrics['recall_macro']:.4f}")
            print(f"F1-score macro       : {val_metrics['f1_macro']:.4f}")
            print(f"Time                 : {elapsed:.2f} sec")

        except Exception as e:
            print(f"{feature_type}__{name} failed: {e}")

    results_df = pd.DataFrame(results)

    if len(results_df) > 0:
        results_df = results_df.sort_values(
            "validation_accuracy",
            ascending=False
        ).reset_index(drop=True)

    return results_df, fitted_models, predictions


print("\n==== Base 8 Models ====")
for name in get_base_8_models():
    print("-", name)


# TRAIN RAW, ROW AVERAGE, AND SVD MODELS

print("\n\n==============================")
print("Training RAW models")
print("==============================")

base8_raw_results, base8_raw_fitted, base8_raw_preds = train_and_evaluate_models(
    models=get_base_8_models(),
    X_train=X_train_ml,
    X_val=X_val_ml,
    y_train=y_train_ml,
    y_val=y_val_ml,
    feature_type="raw"
)

print("\n\n==============================")
print("Training ROW AVERAGE models")
print("==============================")

base8_row_results, base8_row_fitted, base8_row_preds = train_and_evaluate_models(
    models=get_base_8_models(),
    X_train=X_train_row,
    X_val=X_val_row,
    y_train=y_train_ml,
    y_val=y_val_ml,
    feature_type="row_average"
)

print("\n\n==============================")
print("Training SVD models")
print("==============================")

base_8_svd_models = {}
svd_components = min(5, X_train_ml.shape[1] - 1)

for name, model in get_base_8_models().items():
    base_8_svd_models[name] = Pipeline([
        ("svd", TruncatedSVD(n_components=svd_components, random_state=RANDOM_STATE)),
        ("model", model)
    ])

base8_svd_results, base8_svd_fitted, base8_svd_preds = train_and_evaluate_models(
    models=base_8_svd_models,
    X_train=X_train_ml,
    X_val=X_val_ml,
    y_train=y_train_ml,
    y_val=y_val_ml,
    feature_type="svd"
)


# COMBINE RESULTS AND PLOT MODEL PERFORMANCE

base8_all_results = pd.concat([
    base8_raw_results,
    base8_row_results,
    base8_svd_results
], ignore_index=True)

base8_all_results = base8_all_results.sort_values(
    "validation_accuracy",
    ascending=False
).reset_index(drop=True)

base8_all_fitted = {}
base8_all_fitted.update(base8_raw_fitted)
base8_all_fitted.update(base8_row_fitted)
base8_all_fitted.update(base8_svd_fitted)

base8_all_preds = {}
base8_all_preds.update(base8_raw_preds)
base8_all_preds.update(base8_row_preds)
base8_all_preds.update(base8_svd_preds)

print("\n==== All Model Results ====")
print("Total trained models:", len(base8_all_fitted))
display(base8_all_results)

top_base8_models_df = base8_all_results.head(TOP_N_MODELS).copy()

print("\n==== Selected Top Models ====")
display(top_base8_models_df)


top_plot_df = base8_all_results.head(10).copy()

plt.figure(figsize=(15, 7))
x = np.arange(len(top_plot_df))
width = 0.2

plt.bar(x - 1.5 * width, top_plot_df["validation_accuracy"], width, label="Accuracy")
plt.bar(x - 0.5 * width, top_plot_df["precision_macro"], width, label="Precision")
plt.bar(x + 0.5 * width, top_plot_df["recall_macro"], width, label="Recall")
plt.bar(x + 1.5 * width, top_plot_df["f1_macro"], width, label="F1-score")

plt.xticks(x, top_plot_df["model_name"], rotation=75, ha="right")
plt.ylabel("Score")
plt.xlabel("Model")
plt.title("Top 10 Model Performance Comparison")
plt.ylim(0, 1)
plt.legend()
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/top10_model_performance_comparison.png", dpi=300, bbox_inches="tight")
plt.show()


plt.figure(figsize=(15, 6))
plt.bar(top_plot_df["model_name"], top_plot_df["validation_error_rate"])
plt.xticks(rotation=75, ha="right")
plt.ylabel("Error Rate")
plt.xlabel("Model")
plt.title("Top 10 Model Validation Error Rate")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/top10_model_error_rate.png", dpi=300, bbox_inches="tight")
plt.show()


plt.figure(figsize=(15, 6))
plt.plot(top_plot_df["model_name"], top_plot_df["train_accuracy"], marker="o", label="Train Accuracy")
plt.plot(top_plot_df["model_name"], top_plot_df["validation_accuracy"], marker="o", label="Validation Accuracy")
plt.xticks(rotation=75, ha="right")
plt.ylabel("Accuracy")
plt.xlabel("Model")
plt.title("Train vs Validation Accuracy Curve")
plt.ylim(0, 1)
plt.legend()
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/train_vs_validation_accuracy_curve.png", dpi=300, bbox_inches="tight")
plt.show()


def plot_mlp_loss_curve(fitted_models):
    plotted = False

    for model_name, model in fitted_models.items():
        if "MLP" in model_name:
            try:
                mlp_model = model

                if isinstance(model, Pipeline):
                    mlp_model = model.named_steps["clf"]

                if hasattr(mlp_model, "loss_curve_"):
                    plt.figure(figsize=(8, 5))
                    plt.plot(mlp_model.loss_curve_, marker="o")
                    plt.xlabel("Iteration")
                    plt.ylabel("Training Loss")
                    plt.title(f"Loss Curve: {model_name}")
                    plt.grid(True)
                    plt.tight_layout()
                    safe_name = model_name.replace("__", "_")
                    plt.savefig(f"{OUTPUT_DIR}/{safe_name}_loss_curve.png", dpi=300, bbox_inches="tight")
                    plt.show()
                    plotted = True

            except Exception as e:
                print(f"Could not plot MLP loss for {model_name}: {e}")

    if not plotted:
        print("No MLP loss curve found.")


plot_mlp_loss_curve(base8_all_fitted)


# WEIGHTED ENSEMBLE, TOP-K ACCURACY, CONFUSION MATRIX

class Base8ModelWrapper:
    def __init__(self, model_name, model, feature_type, validation_accuracy):
        self.model_name = model_name
        self.model = model
        self.feature_type = feature_type
        self.validation_accuracy = validation_accuracy

    def transform(self, X):
        if self.feature_type == "raw":
            return X

        if self.feature_type == "row_average":
            return row_average_features(X)

        if self.feature_type == "svd":
            return X

        raise ValueError(f"Unknown feature type: {self.feature_type}")

    def predict(self, X):
        Xt = self.transform(X)
        return self.model.predict(Xt)

    def predict_proba(self, X):
        Xt = self.transform(X)
        return safe_predict_proba(
            self.model,
            Xt,
            num_classes=len(label_encoder.classes_)
        )


top_base8_wrapped_models = []

for _, row in top_base8_models_df.iterrows():
    model_name = row["model_name"]
    feature_type = row["feature_type"]
    model = base8_all_fitted[model_name]

    wrapped = Base8ModelWrapper(
        model_name=model_name,
        model=model,
        feature_type=feature_type,
        validation_accuracy=row["validation_accuracy"]
    )

    top_base8_wrapped_models.append(wrapped)

print("\n==== Models Used in Ensemble ====")
for model in top_base8_wrapped_models:
    print(model.model_name, "| acc:", model.validation_accuracy)


def weighted_ensemble_predict_proba(wrapped_models, X):
    weights = np.array([m.validation_accuracy for m in wrapped_models])
    weights = weights / weights.sum()

    final_proba = None

    for model, weight in zip(wrapped_models, weights):
        proba = model.predict_proba(X)

        if final_proba is None:
            final_proba = weight * proba
        else:
            final_proba += weight * proba

    return final_proba


def weighted_ensemble_predict(wrapped_models, X):
    proba = weighted_ensemble_predict_proba(wrapped_models, X)
    return np.argmax(proba, axis=1)


ensemble_val_pred = weighted_ensemble_predict(top_base8_wrapped_models, X_val_ml)
ensemble_val_proba = weighted_ensemble_predict_proba(top_base8_wrapped_models, X_val_ml)

ensemble_metrics = calculate_metrics(y_val_ml, ensemble_val_pred)

print("\n==== Ensemble Validation Performance ====")
print("Best single model accuracy:", base8_all_results.iloc[0]["validation_accuracy"])
print("Weighted ensemble accuracy:", ensemble_metrics["accuracy"])
print("Weighted ensemble error rate:", ensemble_metrics["error_rate"])
print("Precision macro:", ensemble_metrics["precision_macro"])
print("Recall macro:", ensemble_metrics["recall_macro"])
print("F1-score macro:", ensemble_metrics["f1_macro"])
print("Precision weighted:", ensemble_metrics["precision_weighted"])
print("Recall weighted:", ensemble_metrics["recall_weighted"])
print("F1-score weighted:", ensemble_metrics["f1_weighted"])


def top_k_accuracy(y_true, proba, k=3):
    top_k = np.argsort(proba, axis=1)[:, -k:]
    correct = 0

    for true_label, candidates in zip(y_true, top_k):
        if true_label in candidates:
            correct += 1

    return correct / len(y_true)


print("\n==== Ensemble Top-k Accuracy ====")
topk_results = []

for k in [1, 2, 3, 5]:
    acc = top_k_accuracy(y_val_ml, ensemble_val_proba, k=k)
    topk_results.append({"k": k, "top_k_accuracy": acc})
    print(f"Top-{k} accuracy: {acc:.4f}")

topk_df = pd.DataFrame(topk_results)
display(topk_df)


target_names = label_encoder.inverse_transform(
    np.arange(len(label_encoder.classes_))
)

print("\n==== Classification Report: Weighted Ensemble ====")
print(
    classification_report(
        y_val_ml,
        ensemble_val_pred,
        target_names=target_names,
        zero_division=0
    )
)


cm = confusion_matrix(y_val_ml, ensemble_val_pred)

fig, ax = plt.subplots(figsize=(12, 10))

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=target_names
)

disp.plot(
    cmap="Blues",
    xticks_rotation=90,
    values_format="d",
    ax=ax
)

plt.title("Confusion Matrix of Weighted ML Ensemble")
plt.xlabel("Predicted Amino Acid")
plt.ylabel("True Amino Acid")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/weighted_ensemble_confusion_matrix.png", dpi=300, bbox_inches="tight")
plt.show()


class_support = cm.sum(axis=1)
class_correct = np.diag(cm)
class_error = class_support - class_correct
class_accuracy = class_correct / np.maximum(class_support, 1)
class_error_rate = 1 - class_accuracy

class_error_df = pd.DataFrame({
    "amino_acid": target_names,
    "support": class_support,
    "correct": class_correct,
    "errors": class_error,
    "class_accuracy": class_accuracy,
    "class_error_rate": class_error_rate
}).sort_values("class_error_rate", ascending=False)

print("\n==== Class-wise Error Analysis ====")
display(class_error_df)

plt.figure(figsize=(12, 5))
plt.bar(class_error_df["amino_acid"], class_error_df["class_error_rate"])
plt.xlabel("Amino Acid")
plt.ylabel("Class Error Rate")
plt.title("Class-wise Error Rate")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/class_wise_error_rate.png", dpi=300, bbox_inches="tight")
plt.show()


# KNOWN-SIZE GAP PREDICTION WITH BEAM SEARCH

def predict_masked_kmer_topk(masked_kmer, top_k=5):
    x = np.array(encode_sequence(masked_kmer)).reshape(1, -1)

    proba = weighted_ensemble_predict_proba(
        top_base8_wrapped_models,
        x
    )[0]

    top_ids = np.argsort(proba)[::-1][:top_k]

    candidates = []

    for idx in top_ids:
        aa = label_encoder.inverse_transform([idx])[0]
        candidates.append({
            "amino_acid": aa,
            "confidence": float(proba[idx])
        })

    return candidates


def predict_masked_kmer(masked_kmer):
    candidates = predict_masked_kmer_topk(masked_kmer, top_k=5)
    best = candidates[0]
    return best["amino_acid"], best["confidence"], candidates


def right_context_score(candidate, right_context, k=KMER_SIZE):
    if len(candidate) == 0 or len(right_context) == 0:
        return 0.0

    if k % 2 == 0:
        raise ValueError("KMER_SIZE should be odd.")

    scores = []
    half = (k - 1) // 2

    for i in range(len(candidate)):
        prefix = candidate[:i]
        suffix = candidate[i + 1:] + right_context

        left_window = prefix[-half:].rjust(half, GAP_TOKEN)
        right_window = suffix[:half].ljust(half, GAP_TOKEN)

        masked_kmer = left_window + GAP_TOKEN + right_window

        x = np.array(encode_sequence(masked_kmer)).reshape(1, -1)
        proba = weighted_ensemble_predict_proba(top_base8_wrapped_models, x)[0]

        aa = candidate[i]

        if aa in label_encoder.classes_:
            aa_id = label_encoder.transform([aa])[0]
            scores.append(float(proba[aa_id]))

    return float(np.mean(scores)) if scores else 0.0


def fill_known_size_gap_ml(
    left_context,
    right_context,
    gap_size,
    k=KMER_SIZE,
    beam_width=BEAM_WIDTH,
    residue_top_k=RESIDUE_TOP_K
):
    beams = [("", 1.0)]

    for pos in range(gap_size):
        new_beams = []

        for partial_seq, partial_score in beams:
            prefix = (left_context + partial_seq)[-(k - 1):]
            prefix = prefix.rjust(k - 1, GAP_TOKEN)

            masked_kmer = prefix + GAP_TOKEN

            top_candidates = predict_masked_kmer_topk(
                masked_kmer,
                top_k=residue_top_k
            )

            for item in top_candidates:
                aa = item["amino_acid"]
                conf = item["confidence"]

                new_seq = partial_seq + aa
                new_score = partial_score * max(conf, 1e-8)

                new_beams.append((new_seq, new_score))

        new_beams = sorted(new_beams, key=lambda x: x[1], reverse=True)
        beams = new_beams[:beam_width]

    reranked = []

    for seq, left_score in beams:
        rc_score = right_context_score(seq, right_context, k=k)

        final_score = (
            0.70 * math.log(left_score + 1e-8) +
            0.30 * rc_score
        )

        reranked.append({
            "prediction": seq,
            "left_score": left_score,
            "right_context_score": rc_score,
            "final_score": final_score,
            "mass": peptide_mass(seq)
        })

    reranked_df = pd.DataFrame(reranked).sort_values(
        "final_score",
        ascending=False
    ).reset_index(drop=True)

    best = reranked_df.iloc[0]

    return {
        "prediction": best["prediction"],
        "confidence": float(best["final_score"]),
        "mass": int(best["mass"]),
        "top_candidates": reranked_df.to_dict("records")
    }


known_size_example = fill_known_size_gap_ml(
    left_context="DIQMTQSPSSL",
    right_context="SASVGDRVTIT",
    gap_size=3
)

print("\n==== Known-size Example ====")
print(known_size_example)


# GENERATE AND EVALUATE KNOWN-SIZE GAP DATASET

def create_gap_sample(sequence, protein_id, gap_start, gap_len, context_size=15):
    gap_end = gap_start + gap_len
    gap_seq = sequence[gap_start:gap_end]

    left_context = sequence[max(0, gap_start - context_size):gap_start]
    right_context = sequence[gap_end:gap_end + context_size]

    return {
        "protein_id": protein_id,
        "full_sequence": sequence,
        "gap_start": gap_start,
        "gap_end": gap_end,
        "gap_size": gap_len,
        "gap_sequence": gap_seq,
        "gap_mass": peptide_mass(gap_seq),
        "left_context": left_context,
        "right_context": right_context
    }


def generate_gap_dataset(
    sequences,
    protein_prefix,
    samples_per_sequence=5,
    min_gap=1,
    max_gap=8,
    context_size=15
):
    rows = []

    for idx, seq in enumerate(sequences):
        if len(seq) < max_gap + 2 * context_size + 5:
            continue

        for _ in range(samples_per_sequence):
            gap_len = random.randint(min_gap, max_gap)

            min_start = context_size
            max_start = len(seq) - gap_len - context_size

            if max_start <= min_start:
                continue

            gap_start = random.randint(min_start, max_start)

            rows.append(
                create_gap_sample(
                    sequence=seq,
                    protein_id=f"{protein_prefix}_{idx}",
                    gap_start=gap_start,
                    gap_len=gap_len,
                    context_size=context_size
                )
            )

    return pd.DataFrame(rows)


gap_df = pd.concat([
    generate_gap_dataset(mab_train_seqs, "MAB", SAMPLES_PER_SEQUENCE),
    generate_gap_dataset(p5a_train_seqs, "P5A", SAMPLES_PER_SEQUENCE),
    generate_gap_dataset(cah2_train_seqs, "CAH2", SAMPLES_PER_SEQUENCE)
], ignore_index=True)

print("\n==== Known-size Gap Evaluation Samples ====")
print("Gap evaluation samples:", gap_df.shape)
display(gap_df.head())


def residue_accuracy(pred, truth):
    if len(pred) == 0 or len(truth) == 0:
        return 0.0

    n = min(len(pred), len(truth))
    matches = sum(pred[i] == truth[i] for i in range(n))
    return matches / max(len(pred), len(truth))


def evaluate_known_size_gap_filling(df, max_rows=1000):
    rows = []

    eval_df = df.head(max_rows).copy()

    for _, row in eval_df.iterrows():
        result = fill_known_size_gap_ml(
            left_context=row["left_context"],
            right_context=row["right_context"],
            gap_size=int(row["gap_size"]),
            k=KMER_SIZE,
            beam_width=BEAM_WIDTH,
            residue_top_k=RESIDUE_TOP_K
        )

        pred = result["prediction"]
        truth = row["gap_sequence"]

        bio_metrics = biochemical_validation_metrics(pred, truth)

        out = {
            "protein_id": row["protein_id"],
            "truth": truth,
            "prediction": pred,
            "gap_size": row["gap_size"],
            "truth_mass": row["gap_mass"],
            "pred_mass": result["mass"],
            "confidence": result["confidence"],
            "exact_match": pred == truth,
            "residue_accuracy": residue_accuracy(pred, truth)
        }

        out.update(bio_metrics)
        rows.append(out)

    return pd.DataFrame(rows)


known_size_eval_df = evaluate_known_size_gap_filling(
    gap_df,
    max_rows=min(GAP_EVAL_ROWS, len(gap_df))
)

print("\n==== Known-size Evaluation with Biological Metrics ====")
display(known_size_eval_df.head(20))

known_size_exact_acc = known_size_eval_df["exact_match"].mean()
known_size_residue_acc = known_size_eval_df["residue_accuracy"].mean()
known_size_mass_mae = known_size_eval_df["mass_error"].mean()
known_size_hydro_diff = known_size_eval_df["hydrophobicity_diff"].mean()
known_size_charge_diff = known_size_eval_df["charge_diff"].mean()
known_size_comp_sim = known_size_eval_df["composition_similarity"].mean()
known_size_blosum = known_size_eval_df["blosum62_similarity"].mean()

print("Known-size exact match accuracy:", known_size_exact_acc)
print("Known-size mean residue accuracy:", known_size_residue_acc)
print("Known-size mean mass absolute error:", known_size_mass_mae)
print("Known-size mean hydrophobicity difference:", known_size_hydro_diff)
print("Known-size mean charge difference:", known_size_charge_diff)
print("Known-size mean composition similarity:", known_size_comp_sim)
print("Known-size mean BLOSUM62 similarity:", known_size_blosum)


plt.figure(figsize=(8, 5))
plt.hist(known_size_eval_df["residue_accuracy"], bins=20)
plt.xlabel("Residue Accuracy")
plt.ylabel("Frequency")
plt.title("Known-size Gap Residue Accuracy Distribution")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/known_size_residue_accuracy_distribution.png", dpi=300, bbox_inches="tight")
plt.show()


plt.figure(figsize=(8, 5))
plt.hist(known_size_eval_df["mass_error"], bins=20)
plt.xlabel("Mass Error")
plt.ylabel("Frequency")
plt.title("Known-size Gap Mass Error Distribution")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/known_size_mass_error_distribution.png", dpi=300, bbox_inches="tight")
plt.show()


plt.figure(figsize=(8, 5))
plt.hist(known_size_eval_df["blosum62_similarity"], bins=20)
plt.xlabel("BLOSUM62 Similarity")
plt.ylabel("Frequency")
plt.title("Known-size Gap BLOSUM62 Similarity Distribution")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/known_size_blosum62_distribution.png", dpi=300, bbox_inches="tight")
plt.show()


plt.figure(figsize=(8, 5))
plt.hist(known_size_eval_df["composition_similarity"], bins=20)
plt.xlabel("Amino Acid Composition Similarity")
plt.ylabel("Frequency")
plt.title("Known-size Amino Acid Composition Similarity")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/known_size_composition_similarity.png", dpi=300, bbox_inches="tight")
plt.show()


# KNOWN-MASS CANDIDATE GENERATION AND HYBRID RERANKING

def generate_mass_candidates_from_homologs(
    homolog_sequences,
    target_mass,
    tolerance=0,
    min_len=None,
    max_len=None
):
    if min_len is None:
        min_len = max(
            1,
            math.floor((target_mass - tolerance) / max(AA_MASS.values()))
        )

    if max_len is None:
        max_len = math.ceil(
            (target_mass + tolerance) / min(AA_MASS.values())
        )

    candidate_counter = Counter()

    for seq in homolog_sequences:
        seq = str(seq).strip().upper()
        n = len(seq)

        for length in range(min_len, max_len + 1):
            if n < length:
                continue

            for i in range(n - length + 1):
                kmer = seq[i:i + length]

                if not is_valid_protein_sequence(kmer):
                    continue

                mass = peptide_mass(kmer)

                if abs(mass - target_mass) <= tolerance:
                    candidate_counter[kmer] += 1

    return candidate_counter


def context_support_score(
    candidate,
    left_context,
    right_context,
    homolog_sequences,
    flank=5
):
    left_context = str(left_context).upper()
    right_context = str(right_context).upper()

    left = left_context[-flank:] if left_context else ""
    right = right_context[:flank] if right_context else ""

    pattern_full = left + candidate + right
    pattern_left = left + candidate
    pattern_right = candidate + right

    full_count = 0
    left_count = 0
    right_count = 0
    candidate_count = 0

    for seq in homolog_sequences:
        seq = str(seq).upper()

        full_count += seq.count(pattern_full)
        left_count += seq.count(pattern_left)
        right_count += seq.count(pattern_right)
        candidate_count += seq.count(candidate)

    score = (
        6.0 * full_count +
        3.0 * left_count +
        3.0 * right_count +
        1.0 * candidate_count
    )

    return float(score)


def probabilistic_rank_candidates(
    target_mass,
    left_context,
    right_context,
    homolog_sequences,
    tolerance=0,
    top_k=100
):
    candidates = generate_mass_candidates_from_homologs(
        homolog_sequences=homolog_sequences,
        target_mass=target_mass,
        tolerance=tolerance
    )

    rows = []

    for candidate, frequency in candidates.items():
        mass = peptide_mass(candidate)
        mass_error = abs(mass - target_mass)

        context_score = context_support_score(
            candidate=candidate,
            left_context=left_context,
            right_context=right_context,
            homolog_sequences=homolog_sequences,
            flank=5
        )

        probabilistic_score = (
            0.5 * frequency +
            2.0 * context_score -
            mass_error
        )

        rows.append({
            "candidate": candidate,
            "candidate_length": len(candidate),
            "candidate_mass": mass,
            "target_mass": target_mass,
            "mass_error": mass_error,
            "homolog_frequency": frequency,
            "context_score": context_score,
            "probabilistic_score": probabilistic_score,
            "mass_valid": mass_error <= tolerance
        })

    df = pd.DataFrame(rows)

    if len(df) == 0:
        return df

    return df.sort_values(
        "probabilistic_score",
        ascending=False
    ).head(top_k).reset_index(drop=True)


def ml_score_candidate(left_context, right_context, candidate, k=KMER_SIZE):
    if len(candidate) == 0:
        return 0.0

    if k % 2 == 0:
        raise ValueError("KMER_SIZE should be odd.")

    scores = []
    half = (k - 1) // 2

    for pos, aa in enumerate(candidate):
        left_part = left_context + candidate[:pos]
        right_part = candidate[pos + 1:] + right_context

        left_window = left_part[-half:].rjust(half, GAP_TOKEN)
        right_window = right_part[:half].ljust(half, GAP_TOKEN)

        masked_kmer = left_window + GAP_TOKEN + right_window

        x = np.array(encode_sequence(masked_kmer)).reshape(1, -1)

        proba = weighted_ensemble_predict_proba(
            top_base8_wrapped_models,
            x
        )[0]

        if aa in label_encoder.classes_:
            aa_id = label_encoder.transform([aa])[0]
            scores.append(float(proba[aa_id]))
        else:
            scores.append(0.0)

    return float(np.mean(scores)) if scores else 0.0


def hybrid_rerank_candidates(
    candidate_df,
    left_context,
    right_context,
    target_mass,
    mass_tolerance=0,
    expected_length=None,
    scoring_mode="full_hybrid"
):
    if candidate_df is None or len(candidate_df) == 0:
        return pd.DataFrame()

    rows = []

    if expected_length is None:
        expected_length = round(target_mass / 110)

    for _, row in candidate_df.iterrows():
        candidate = row["candidate"]

        candidate_mass = peptide_mass(candidate)
        mass_error = abs(candidate_mass - target_mass)

        mass_valid_score = 1.0 if mass_error <= mass_tolerance else 0.0
        homolog_frequency_score = math.log1p(row["homolog_frequency"])
        context_score_norm = math.log1p(row["context_score"])
        length_penalty = abs(len(candidate) - expected_length)

        if scoring_mode in ["mass_context_ml", "full_hybrid"]:
            ml_score = ml_score_candidate(
                left_context=left_context,
                right_context=right_context,
                candidate=candidate,
                k=KMER_SIZE
            )
        else:
            ml_score = 0.0

        if scoring_mode == "mass_only":
            hybrid_score = (
                3.0 * mass_valid_score -
                2.0 * mass_error
            )

        elif scoring_mode == "mass_frequency":
            hybrid_score = (
                3.0 * mass_valid_score +
                2.0 * homolog_frequency_score -
                2.0 * mass_error
            )

        elif scoring_mode == "mass_context":
            hybrid_score = (
                3.0 * mass_valid_score +
                1.0 * homolog_frequency_score +
                4.0 * context_score_norm -
                2.0 * mass_error
            )

        elif scoring_mode == "mass_context_ml":
            hybrid_score = (
                3.0 * mass_valid_score +
                1.0 * homolog_frequency_score +
                4.0 * context_score_norm +
                3.0 * ml_score -
                2.0 * mass_error
            )

        elif scoring_mode == "full_hybrid":
            hybrid_score = (
                3.0 * mass_valid_score +
                1.0 * homolog_frequency_score +
                4.0 * context_score_norm +
                3.0 * ml_score -
                2.0 * mass_error -
                0.5 * length_penalty
            )

        else:
            raise ValueError("Unknown scoring_mode:", scoring_mode)

        out = row.to_dict()
        out.update({
            "scoring_mode": scoring_mode,
            "ml_ensemble_score": ml_score,
            "mass_valid_score": mass_valid_score,
            "expected_length": expected_length,
            "length_penalty": length_penalty,
            "hybrid_score": hybrid_score
        })

        rows.append(out)

    out_df = pd.DataFrame(rows)

    return out_df.sort_values(
        "hybrid_score",
        ascending=False
    ).reset_index(drop=True)


print("\nKnown-mass candidate generation and hybrid reranking functions loaded.")


# CAH2 KNOWN-MASS TEST CASES AND ABLATION STUDY

cah2_test_cases = [
    {
        "gap_name": "CAH2_gap_1",
        "mass": 420,
        "left": "MSHHWGYGKHN",
        "right": "WHKDFPIAKGER",
        "truth": "GPEH"
    },
    {
        "gap_name": "CAH2_gap_2",
        "mass": 750,
        "left": "DPSLKPL",
        "right": "TSLRILNNGHA",
        "truth": "SVSYDQA"
    },
    {
        "gap_name": "CAH2_gap_3",
        "mass": 622,
        "left": "SQDK",
        "right": "LDGTYRLIQF",
        "truth": "AVLKGGP"
    },
    {
        "gap_name": "CAH2_gap_4",
        "mass": 1318,
        "left": "VHWNT",
        "right": "DGLAVLGIFL",
        "truth": "KYGDFGKAVQQP"
    },
    {
        "gap_name": "CAH2_gap_5",
        "mass": 751,
        "left": "DYWTYPGSL",
        "right": "CVTWIVLKEP",
        "truth": "TTPPLLE"
    },
    {
        "gap_name": "CAH2_gap_6",
        "mass": 544,
        "left": "VSSEQV",
        "right": "KLNFNGEGEP",
        "truth": "LKFR"
    },
    {
        "gap_name": "CAH2_gap_7",
        "mass": 561,
        "left": "PLKNRQI",
        "right": "",
        "truth": "KASFK"
    }
]


def evaluate_known_mass_ablation(
    use_truth_length=False,
    label="without_truth_length"
):
    scoring_modes = [
        "mass_only",
        "mass_frequency",
        "mass_context",
        "mass_context_ml",
        "full_hybrid"
    ]

    all_rows = []

    for mode in scoring_modes:
        print("\n======================================")
        print("Ablation mode:", mode, "|", label)
        print("======================================")

        for case in cah2_test_cases:
            prob_candidates = probabilistic_rank_candidates(
                target_mass=case["mass"],
                left_context=case["left"],
                right_context=case["right"],
                homolog_sequences=cah2_train_seqs,
                tolerance=0,
                top_k=100
            )

            expected_length = len(case["truth"]) if use_truth_length else None

            ranked_df = hybrid_rerank_candidates(
                candidate_df=prob_candidates,
                left_context=case["left"],
                right_context=case["right"],
                target_mass=case["mass"],
                mass_tolerance=0,
                expected_length=expected_length,
                scoring_mode=mode
            )

            if len(ranked_df) == 0:
                pred = ""
                pred_mass = None
                rank_of_truth = None
                top_5 = []
            else:
                pred = ranked_df.iloc[0]["candidate"]
                pred_mass = ranked_df.iloc[0]["candidate_mass"]
                top_5 = ranked_df["candidate"].head(5).tolist()

                truth_positions = ranked_df.index[
                    ranked_df["candidate"] == case["truth"]
                ].tolist()

                rank_of_truth = truth_positions[0] + 1 if truth_positions else None

            bio_metrics = biochemical_validation_metrics(pred, case["truth"])

            row = {
                "evaluation_type": label,
                "scoring_mode": mode,
                "gap_name": case["gap_name"],
                "target_mass": case["mass"],
                "truth": case["truth"],
                "prediction": pred,
                "truth_length": len(case["truth"]),
                "prediction_length": len(pred),
                "pred_mass": pred_mass,
                "exact_match": pred == case["truth"],
                "rank_of_truth": rank_of_truth,
                "top_3_recovery": rank_of_truth is not None and rank_of_truth <= 3,
                "top_5_recovery": rank_of_truth is not None and rank_of_truth <= 5,
                "top_5_candidates": top_5
            }

            row.update(bio_metrics)
            all_rows.append(row)

    ablation_df = pd.DataFrame(all_rows)

    summary = ablation_df.groupby(["evaluation_type", "scoring_mode"]).agg(
        exact_match_accuracy=("exact_match", "mean"),
        top_3_recovery=("top_3_recovery", "mean"),
        top_5_recovery=("top_5_recovery", "mean"),
        mean_mass_error=("mass_error", "mean"),
        mean_hydrophobicity_diff=("hydrophobicity_diff", "mean"),
        mean_charge_diff=("charge_diff", "mean"),
        mean_composition_similarity=("composition_similarity", "mean"),
        mean_blosum62_similarity=("blosum62_similarity", "mean")
    ).reset_index()

    return ablation_df, summary


known_mass_ablation_fair_df, known_mass_ablation_fair_summary = evaluate_known_mass_ablation(
    use_truth_length=False,
    label="without_truth_length"
)

known_mass_ablation_len_df, known_mass_ablation_len_summary = evaluate_known_mass_ablation(
    use_truth_length=True,
    label="with_known_length"
)

known_mass_ablation_df = pd.concat(
    [known_mass_ablation_fair_df, known_mass_ablation_len_df],
    ignore_index=True
)

known_mass_ablation_summary = pd.concat(
    [known_mass_ablation_fair_summary, known_mass_ablation_len_summary],
    ignore_index=True
)

print("\n==== Known-mass Ablation Summary ====")
display(known_mass_ablation_summary)


plt.figure(figsize=(12, 6))
fair_plot = known_mass_ablation_summary[
    known_mass_ablation_summary["evaluation_type"] == "without_truth_length"
]
plt.bar(fair_plot["scoring_mode"], fair_plot["exact_match_accuracy"])
plt.xlabel("Scoring Mode")
plt.ylabel("Exact Match Accuracy")
plt.title("Known-mass Ablation Study without Truth Length")
plt.xticks(rotation=45, ha="right")
plt.ylim(0, 1)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/known_mass_ablation_without_truth_length.png", dpi=300, bbox_inches="tight")
plt.show()


plt.figure(figsize=(12, 6))
len_plot = known_mass_ablation_summary[
    known_mass_ablation_summary["evaluation_type"] == "with_known_length"
]
plt.bar(len_plot["scoring_mode"], len_plot["exact_match_accuracy"])
plt.xlabel("Scoring Mode")
plt.ylabel("Exact Match Accuracy")
plt.title("Known-mass Ablation Study with Known Gap Length")
plt.xticks(rotation=45, ha="right")
plt.ylim(0, 1)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/known_mass_ablation_with_known_length.png", dpi=300, bbox_inches="tight")
plt.show()


# FINAL INFERENCE FUNCTION

def final_predict_gap(
    left_context,
    right_context,
    known_gap_size=None,
    known_gap_mass=None,
    homolog_sequences=None,
    mass_tolerance=0,
    top_k=10,
    scoring_mode="full_hybrid"
):
    if homolog_sequences is None:
        homolog_sequences = all_homologs

    if known_gap_mass is not None:
        prob_candidates = probabilistic_rank_candidates(
            target_mass=known_gap_mass,
            left_context=left_context,
            right_context=right_context,
            homolog_sequences=homolog_sequences,
            tolerance=mass_tolerance,
            top_k=100
        )

        if known_gap_size is not None and len(prob_candidates) > 0:
            prob_candidates = prob_candidates[
                prob_candidates["candidate_length"] == known_gap_size
            ].reset_index(drop=True)

        expected_length = known_gap_size if known_gap_size is not None else None

        hybrid_candidates = hybrid_rerank_candidates(
            candidate_df=prob_candidates,
            left_context=left_context,
            right_context=right_context,
            target_mass=known_gap_mass,
            mass_tolerance=mass_tolerance,
            expected_length=expected_length,
            scoring_mode=scoring_mode
        )

        if len(hybrid_candidates) == 0:
            return {
                "mode": "known_mass_hybrid",
                "prediction": "",
                "confidence": 0.0,
                "top_candidates": []
            }

        best = hybrid_candidates.iloc[0]

        return {
            "mode": "known_mass_hybrid",
            "scoring_mode": scoring_mode,
            "prediction": best["candidate"],
            "confidence": float(best["hybrid_score"]),
            "mass_valid": bool(best["mass_valid"]),
            "pred_mass": int(best["candidate_mass"]),
            "target_mass": int(known_gap_mass),
            "expected_length": int(best["expected_length"]),
            "length_penalty": float(best["length_penalty"]),
            "top_candidates": hybrid_candidates.head(top_k).to_dict("records")
        }

    if known_gap_size is not None:
        result = fill_known_size_gap_ml(
            left_context=left_context,
            right_context=right_context,
            gap_size=known_gap_size,
            k=KMER_SIZE,
            beam_width=BEAM_WIDTH,
            residue_top_k=RESIDUE_TOP_K
        )

        return {
            "mode": "known_size_beam_search_ml_ensemble",
            "prediction": result["prediction"],
            "confidence": result["confidence"],
            "pred_mass": result["mass"],
            "top_candidates": result["top_candidates"]
        }

    raise ValueError("Provide known_gap_size or known_gap_mass.")


print("\n==== Final Inference Tests ====")

known_size_result = final_predict_gap(
    left_context="DIQMTQSPSSL",
    right_context="SASVGDRVTIT",
    known_gap_size=3
)

print("Known-size result:")
print(known_size_result)

known_mass_result = final_predict_gap(
    left_context="MSHHWGYGKHN",
    right_context="WHKDFPIAKGER",
    known_gap_mass=420,
    known_gap_size=4,
    homolog_sequences=cah2_train_seqs,
    mass_tolerance=0,
    top_k=5,
    scoring_mode="full_hybrid"
)

print("\nKnown-mass result:")
print("Prediction:", known_mass_result["prediction"])
print("Mass valid:", known_mass_result["mass_valid"])
display(pd.DataFrame(known_mass_result["top_candidates"]).head())



# SUMMARY AND SAVE FILES

best_ablation_fair = known_mass_ablation_fair_summary.sort_values(
    "exact_match_accuracy",
    ascending=False
).iloc[0]

best_ablation_len = known_mass_ablation_len_summary.sort_values(
    "exact_match_accuracy",
    ascending=False
).iloc[0]

summary_df = pd.DataFrame([
    {
        "system": "Best single base model",
        "task": "Residue-level amino acid prediction",
        "accuracy": base8_all_results.iloc[0]["validation_accuracy"],
        "error_rate": base8_all_results.iloc[0]["validation_error_rate"],
        "precision": base8_all_results.iloc[0]["precision_macro"],
        "recall": base8_all_results.iloc[0]["recall_macro"],
        "f1_score": base8_all_results.iloc[0]["f1_macro"],
        "biological_metric": None
    },
    {
        "system": f"Weighted ensemble top {TOP_N_MODELS}",
        "task": "Residue-level amino acid prediction",
        "accuracy": ensemble_metrics["accuracy"],
        "error_rate": ensemble_metrics["error_rate"],
        "precision": ensemble_metrics["precision_macro"],
        "recall": ensemble_metrics["recall_macro"],
        "f1_score": ensemble_metrics["f1_macro"],
        "biological_metric": None
    },
    {
        "system": "Known-size beam-search ML ensemble",
        "task": "Full gap sequence prediction",
        "accuracy": known_size_exact_acc,
        "error_rate": 1.0 - known_size_exact_acc,
        "precision": None,
        "recall": None,
        "f1_score": None,
        "biological_metric": "Exact match"
    },
    {
        "system": "Known-size beam-search ML ensemble",
        "task": "Residue-level gap similarity",
        "accuracy": known_size_residue_acc,
        "error_rate": 1.0 - known_size_residue_acc,
        "precision": None,
        "recall": None,
        "f1_score": None,
        "biological_metric": "Residue similarity"
    },
    {
        "system": "Known-size beam-search ML ensemble",
        "task": "Biochemical plausibility",
        "accuracy": known_size_comp_sim,
        "error_rate": None,
        "precision": None,
        "recall": None,
        "f1_score": None,
        "biological_metric": "Amino acid composition similarity"
    },
    {
        "system": "Known-size beam-search ML ensemble",
        "task": "Biochemical plausibility",
        "accuracy": known_size_blosum,
        "error_rate": None,
        "precision": None,
        "recall": None,
        "f1_score": None,
        "biological_metric": "Mean BLOSUM62 similarity"
    },
    {
        "system": "Best known-mass ablation model",
        "task": "Mass-guided prediction without truth length",
        "accuracy": best_ablation_fair["exact_match_accuracy"],
        "error_rate": 1.0 - best_ablation_fair["exact_match_accuracy"],
        "precision": None,
        "recall": None,
        "f1_score": None,
        "biological_metric": best_ablation_fair["scoring_mode"]
    },
    {
        "system": "Best known-mass ablation model",
        "task": "Top-5 recovery without truth length",
        "accuracy": best_ablation_fair["top_5_recovery"],
        "error_rate": 1.0 - best_ablation_fair["top_5_recovery"],
        "precision": None,
        "recall": None,
        "f1_score": None,
        "biological_metric": best_ablation_fair["scoring_mode"]
    },
    {
        "system": "Best known-mass ablation model + known length",
        "task": "Mass-guided prediction with known gap length",
        "accuracy": best_ablation_len["exact_match_accuracy"],
        "error_rate": 1.0 - best_ablation_len["exact_match_accuracy"],
        "precision": None,
        "recall": None,
        "f1_score": None,
        "biological_metric": best_ablation_len["scoring_mode"]
    },
    {
        "system": "Best known-mass ablation model + known length",
        "task": "Top-5 recovery with known gap length",
        "accuracy": best_ablation_len["top_5_recovery"],
        "error_rate": 1.0 - best_ablation_len["top_5_recovery"],
        "precision": None,
        "recall": None,
        "f1_score": None,
        "biological_metric": best_ablation_len["scoring_mode"]
    }
])

print("\n==== Final Publication-Level Performance Summary ====")
display(summary_df)


base8_all_results.to_csv(f"{OUTPUT_DIR}/base8_all_model_results_with_metrics.csv", index=False)
top_base8_models_df.to_csv(f"{OUTPUT_DIR}/base8_selected_top_models.csv", index=False)
topk_df.to_csv(f"{OUTPUT_DIR}/ensemble_topk_accuracy.csv", index=False)
class_error_df.to_csv(f"{OUTPUT_DIR}/class_wise_error_analysis.csv", index=False)

known_size_eval_df.to_csv(
    f"{OUTPUT_DIR}/known_size_gap_evaluation_with_biochemical_metrics.csv",
    index=False
)

known_mass_ablation_df.to_csv(
    f"{OUTPUT_DIR}/known_mass_ablation_full_results.csv",
    index=False
)

known_mass_ablation_summary.to_csv(
    f"{OUTPUT_DIR}/known_mass_ablation_summary.csv",
    index=False
)

summary_df.to_csv(
    f"{OUTPUT_DIR}/final_publication_level_performance_summary.csv",
    index=False
)

print("\n==== Saved Files ====")
for file in os.listdir(OUTPUT_DIR):
    if file.endswith(".csv") or file.endswith(".png"):
        print(os.path.join(OUTPUT_DIR, file))


best_model_name = base8_all_results.iloc[0]["model_name"]
best_model_acc = base8_all_results.iloc[0]["validation_accuracy"]
best_model_f1 = base8_all_results.iloc[0]["f1_macro"]

text = f"""
The experimental results demonstrate that the proposed hybrid protein scaffold gap-filling framework achieved strong predictive performance across residue-level and gap-level tasks. Among the evaluated machine learning models, {best_model_name} achieved the highest validation accuracy of {best_model_acc:.4f} with a macro F1-score of {best_model_f1:.4f}. The weighted ensemble constructed from the top {TOP_N_MODELS} models obtained an accuracy of {ensemble_metrics['accuracy']:.4f}, macro precision of {ensemble_metrics['precision_macro']:.4f}, macro recall of {ensemble_metrics['recall_macro']:.4f}, and macro F1-score of {ensemble_metrics['f1_macro']:.4f}.

For known-size gap filling, the beam-search ML ensemble achieved an exact match accuracy of {known_size_exact_acc:.4f} and a residue-level similarity of {known_size_residue_acc:.4f}. Biochemical validation further showed an amino acid composition similarity of {known_size_comp_sim:.4f} and a mean BLOSUM62 similarity of {known_size_blosum:.4f}. For mass-guided gap prediction without using the true gap length, the best ablation configuration was {best_ablation_fair['scoring_mode']}, achieving an exact match accuracy of {best_ablation_fair['exact_match_accuracy']:.4f} and top-5 recovery of {best_ablation_fair['top_5_recovery']:.4f}. When known gap length was additionally provided, the best configuration was {best_ablation_len['scoring_mode']}, achieving an exact match accuracy of {best_ablation_len['exact_match_accuracy']:.4f}. These findings demonstrate that the integration of sequence-context learning, homologous peptide retrieval, mass constraints, beam-search decoding, biological validation, and hybrid reranking provides an effective solution for protein scaffold gap filling.
"""

print("\n==== Manuscript-ready Result Text ====")
print(text)